# Import software libraries

In [1]:
# Import required libraries.
import sys           # Read system parameters.
import pandas as pd  # Manipulate and analyze data.
import sqlite3       # Manage SQL databases.

# Summarize software libraries used.
print('Libraries used in this project:')
print('- Python {}'.format(sys.version))
print('- pandas {}'.format(pd.__version__))
print('- sqlite3 {}'.format(sqlite3.sqlite_version))

Libraries used in this project:
- Python 3.7.6 | packaged by conda-forge | (default, Mar 23 2020, 23:03:20) 
[GCC 7.3.0]
- pandas 1.1.3
- sqlite3 3.30.1


# Examine the database

In [2]:
# Connect to SQLite database.
conn = sqlite3.connect('data/prod_sample.db')


In [10]:
# List all the tables in the database.
query = 'SELECT name FROM sqlite_master WHERE type = "table"'

pd.read_sql(query, conn)

,name
0,stock_description
1,online_retail_history


# Read data from the `online_retail_history` table

In [11]:
# Write the query to be executed that selects everything from the online_retail_history table.
query = 'SELECT * FROM online_retail_history'



# Use the read_sql function in pandas to read a query into a DataFrame.
history = pd.read_sql(query, conn)


# Preview the first five rows of the data.
history.head()


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50


In [12]:
# Get the shape of the data.
history.shape


(15321, 8)

# Read data from the `stock_description` table

In [13]:
# Write the query to be executed that selects everything from the stock_description table.
query = 'SELECT * FROM stock_description'



# Use the read_sql function in pandas to read a query into a DataFrame.
stock = pd.read_sql(query, conn)


# Preview the first five rows of the data.
stock.head()



,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10080,GROOVY CACTUS INFLATABLE
2,10120,DOGGY RUBBER
3,10123C,HEARTS WRAPPING TAPE
4,10124A,SPOTS ON RED BOOKCOVER TAPE


In [14]:
# Get the shape of the data.
stock.shape


(3952, 2)

# Aggregate the `online_retail_history` and `stock_description` datasets

In [17]:
# Write a query to aggregate the two datasets so that you have the stock descriptions as well as the stock code.
query = '''SELECT hi.invoice, hi.stockcode, hi.quantity, hi.invoicedate, hi.price, hi.customerID, hi.country, hi.totalamount, st.description 
        FROM online_retail_history AS hi LEFT JOIN stock_description AS st
        WHERE hi.stockcode = st.stockcode
        '''






# Use the read_sql function in pandas to read a query into a DataFrame.
agg = pd.read_sql(query, conn)


# Preview the first five rows of the data.
agg.head()


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,LUNCH BAG RED RETROSPOT


In [24]:
# Get the shape of the data.
agg.shape
agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17032 entries, 0 to 17031
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Invoice      17032 non-null  object 
 1   StockCode    17032 non-null  object 
 2   Quantity     17032 non-null  int64  
 3   InvoiceDate  17032 non-null  object 
 4   Price        17019 non-null  float64
 5   CustomerID   13976 non-null  object 
 6   Country      17032 non-null  object 
 7   TotalAmount  17019 non-null  float64
 8   Description  17032 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 1.2+ MB


# Identify and fix corrupt or unusable data

In [25]:
# Check the value counts of the "Description" field.
agg.Description.value_counts

<bound method IndexOpsMixin.value_counts of 0        CREAM HANGING HEART T-LIGHT HOLDER
1             ASSORTED COLOUR BIRD ORNAMENT
2        CREAM HANGING HEART T-LIGHT HOLDER
3        CREAM HANGING HEART T-LIGHT HOLDER
4                   LUNCH BAG RED RETROSPOT
                        ...                
17027               LUNCH BAG  BLACK SKULL.
17028               LUNCH BAG RED RETROSPOT
17029                        POPCORN HOLDER
17030               LUNCH BAG RED RETROSPOT
17031         ASSORTED COLOUR BIRD ORNAMENT
Name: Description, Length: 17032, dtype: object>

In [29]:
# Remove rows where "Description" is just a question mark (?).
agg.drop(agg.index[(agg['Description'] == '?')], axis=0, inplace=True)






# Preview the first five rows of the data.
agg.shape


(15321, 9)

# Identify and remove duplicates

In [31]:
# Identify all duplicated data.
duplicated_data = \
agg[agg.duplicated(keep = False)]



In [32]:
# Print the duplicated data.
print(duplicated_data.shape[0])


223


In [36]:
# Remove the duplicated data.
agg_clean = \
agg[~agg.duplicated()]






# Preview the first five rows of the data.
agg_clean.head()

,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,LUNCH BAG RED RETROSPOT


# Correct date formats

In [37]:
# Get the data types for every column in the DataFrame.
agg_clean.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 15206 entries, 0 to 17031
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Invoice      15206 non-null  object 
 1   StockCode    15206 non-null  object 
 2   Quantity     15206 non-null  int64  
 3   InvoiceDate  15206 non-null  object 
 4   Price        15194 non-null  float64
 5   CustomerID   12435 non-null  object 
 6   Country      15206 non-null  object 
 7   TotalAmount  15194 non-null  float64
 8   Description  15206 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 1.2+ MB


In [40]:
# Convert "InvoiceDate" to a "%Y-%m-%d" datetime format.
agg_cleaned_2 = agg_clean.copy()
agg_cleaned_2['InvoiceDate'] = \
pd.to_datetime(agg_cleaned_2['InvoiceDate'], format = '%Y-%m-%d')



In [42]:
# Get the data types for every column in the converted DataFrame.
agg_cleaned_2.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 15206 entries, 0 to 17031
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Invoice      15206 non-null  object        
 1   StockCode    15206 non-null  object        
 2   Quantity     15206 non-null  int64         
 3   InvoiceDate  15206 non-null  datetime64[ns]
 4   Price        15194 non-null  float64       
 5   CustomerID   12435 non-null  object        
 6   Country      15206 non-null  object        
 7   TotalAmount  15194 non-null  float64       
 8   Description  15206 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 1.2+ MB


# Examine the table before finishing

In [43]:
# Preview the first five rows of the data.
agg_cleaned_2.head()


,Invoice,StockCode,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount,Description
0,536365,85123A,6,2010-12-01 08:26:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
1,536367,84879,32,2010-12-01 08:34:00,1.69,u13047,United Kingdom,54.08,ASSORTED COLOUR BIRD ORNAMENT
2,536373,85123A,6,2010-12-01 09:02:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
3,536375,85123A,6,2010-12-01 09:32:00,2.55,u1785,United Kingdom,15.30,CREAM HANGING HEART T-LIGHT HOLDER
4,536378,20725,10,2010-12-01 09:37:00,1.65,u14688,United Kingdom,16.50,LUNCH BAG RED RETROSPOT


# Load the dataset into a pickle file

In [44]:
# Save the dataset as a pickle file named online_history_cleaned.pickle.
agg_cleaned_2.to_pickle('online_history_cleaned.pickle')


In [45]:
# Close any connections to the database.
conn.close()
